# KG1 v155 PARSER TEST - Hipotese Gemini Pro 2nd round

## TESTE: Modelo produz formato \boxed{X} OFF que parser Kaggle perde?

### Como funciona Kaggle submission:
```
Felipe submete: apenas adapter LoRA (.safetensors + adapter_config.json)
Kaggle roda: inferencia DELES + regex DELES (imutavel)
Regex: \boxed\{([^}]*)(?:\}|$)
Verify: float isclose OR string lower compare
```

### Hipotese a testar:
Modelo v120 atual (0.86) acerta ~10 questoes extras mas em formato OFF:
- `\boxed{x = 42}` em vez de `\boxed{42}`
- `\boxed{42 m/s}` em vez de `\boxed{42}`
- `\boxed{42.}` em vez de `\boxed{42}`

### Custo: ~30 min, 0 slots Kaggle

### Decisao apos teste:
- Tipo C > 10%: **PARSER MISS CONFIRMADO** -> SFT format-strict (~200 steps)
- Tipo B > Tipo C: math problem -> training normal
- Tipo A > 20%: max_tokens issue

## Setup
1. Runtime A100 ALTA RAM
2. Colab Secrets: HF_KEY, KAGGLE_USERNAME, KAGGLE_KEY
3. Form param ADAPTER_PATH: caminho seu adapter 0.86 (vazio = scan GDrive auto + fallback huikang)
4. Run all


In [ ]:
# CELL UNICA v155 - Parser miss diagnostic
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess, shutil, re

# === FORM PARAMS ===
ADAPTER_PATH = ""  #@param {type:"string"}
N_SAMPLES = 100  #@param {type:"integer"}
MAX_NEW_TOKENS = 2048  #@param {type:"integer"}

# === ENV ===
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print('=' * 70)
print('KG1 v155 PARSER TEST - testar hipotese Gemini Pro')
print('=' * 70)

# ============================================================
# STEP 1: Setup
# ============================================================
print('\n[1/8] Setup')
try:
    from google.colab import userdata, drive
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing')
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except: pass
except ImportError:
    raise RuntimeError('Run no Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')

# ============================================================
# STEP 2: Install deps minimo
# ============================================================
print('\n[2/8] Install deps')
deps = ['transformers>=4.48.0', 'peft>=0.14.0', 'datasets>=3.0.0',
        'accelerate>=1.3.0', 'bitsandbytes>=0.45.0', 'huggingface_hub>=0.27.0',
        'kaggle', 'einops', 'packaging', 'ninja', 'triton']
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=300)

for m in ['transformers', 'peft', 'bitsandbytes']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# STEP 3: Mamba install
# ============================================================
print('\n[3/8] Install mamba-ssm')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'

attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] {cu} torch{t} abi{abi}')
        break

# ============================================================
# STEP 4: Locate adapter
# ============================================================
print('\n[4/8] Locate adapter')

def find_adapter_dir(root):
    for r, dirs, files in os.walk(root):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            sz = os.path.getsize(os.path.join(r, 'adapter_model.safetensors'))
            if sz > 100_000_000:  # skip stubs <100MB
                return r
    return None

adapter_path = None
adapter_source = None

if ADAPTER_PATH and os.path.exists(ADAPTER_PATH):
    p = ADAPTER_PATH if os.path.isdir(ADAPTER_PATH) else os.path.dirname(ADAPTER_PATH)
    found = find_adapter_dir(p) if os.path.isdir(p) else None
    if found:
        adapter_path = found
        adapter_source = 'form_param'

if not adapter_path:
    try:
        drive.mount('/content/drive', force_remount=False)
        for d in os.listdir('/content/drive/MyDrive'):
            full = f'/content/drive/MyDrive/{d}'
            if os.path.isdir(full) and ('kg1' in d.lower() or 'nemotron' in d.lower() or 'adapter' in d.lower()):
                ap = find_adapter_dir(full)
                if ap:
                    adapter_path = ap
                    adapter_source = f'gdrive_{d}'
                    break
    except Exception as e:
        print(f'  GDrive scan: {e}')

if not adapter_path:
    print('  No local adapter, downloading huikang mirror...')
    HUIKANG_DIR = '/content/huikang_adapter'
    os.makedirs(HUIKANG_DIR, exist_ok=True)
    r = subprocess.run(['kaggle', 'datasets', 'download', '-d',
                       'andreyyunoshev/huikang-nemotron-adapter-mirror',
                       '-p', HUIKANG_DIR, '--unzip'],
                      capture_output=True, text=True, timeout=900)
    adapter_path = find_adapter_dir(HUIKANG_DIR)
    adapter_source = 'huikang_0.85_FALLBACK'

if not adapter_path:
    raise RuntimeError('No adapter found anywhere')

print(f'  ADAPTER: {adapter_path}')
print(f'  SOURCE: {adapter_source}')
sz_mb = os.path.getsize(os.path.join(adapter_path, 'adapter_model.safetensors')) / 1e6
print(f'  Size: {sz_mb:.1f} MB')

# ============================================================
# STEP 5: Load model + adapter
# ============================================================
print('\n[5/8] Load Nemotron-30B + adapter (inference only)')
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

print('  Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('  Loading model BF16 eager...')
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
    trust_remote_code=True, token=hf_token, attn_implementation='eager',
)
model.config.use_cache = True  # inference: cache ON
print(f'  [OK] {time.time()-t0:.1f}s')

print(f'  Loading adapter (inference only)...')
model = PeftModel.from_pretrained(model, adapter_path, adapter_name='default',
                                   token=hf_token, is_trainable=False)
model.eval()
print(f'  VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# ============================================================
# STEP 6: Load test data com gabarito
# ============================================================
print('\n[6/8] Load test data (cryptarithm_813 com gabarito)')

DATA_PATH = '/content/cryptarithm_813.jsonl'
url = 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl'
urllib.request.urlretrieve(url, DATA_PATH)

samples = []
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        row = json.loads(line)
        if row.get('is_valid'):
            samples.append(row)
print(f'  Loaded {len(samples)} samples (using first {N_SAMPLES})')

# ============================================================
# STEP 7: Inference + Kaggle parser test
# ============================================================
print(f'\n[7/8] Inference {N_SAMPLES} samples with KAGGLE EXACT REGEX + VERIFY')

# REGEX KAGGLE EXATO (do project_metric_update.md)
KAGGLE_REGEX = re.compile(r'\\boxed\{([^}]*)(?:\}|$)')

def kaggle_verify(stored_answer, predicted):
    """REPLICA EXATA do verify() Kaggle."""
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()
    if re.fullmatch(r'[01]+', stored_answer):
        return predicted.lower() == stored_answer.lower()
    try:
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

results = {
    'total': 0,
    'kaggle_correct': 0,
    'tipo_a_no_boxed': 0,      # nenhum \boxed{}
    'tipo_b_math_wrong': 0,     # \boxed{} presente, math errado
    'tipo_c_format_off': 0,     # \boxed{} presente, format quase certo
    'tipo_c_examples': [],
}

t_start = time.time()
for i, row in enumerate(samples[:N_SAMPLES]):
    if i % 10 == 0:
        elapsed = time.time() - t_start
        rate = i / elapsed if elapsed > 0 else 0
        eta = (N_SAMPLES - i) / rate if rate > 0 else 0
        print(f'  [{i}/{N_SAMPLES}] elapsed={elapsed:.0f}s rate={rate:.2f}/s ETA={eta:.0f}s')

    prompt_raw = row.get('prompt', '').strip()
    gold = row.get('answer', '').strip()
    if not prompt_raw or not gold:
        continue

    messages = [{'role': 'user', 'content': prompt_raw + PROMPT_SUFFIX}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt',
                                            add_generation_prompt=True,
                                            enable_thinking=True).to('cuda')
    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=MAX_NEW_TOKENS,
                                  temperature=0.0, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    results['total'] += 1

    # Aplicar regex Kaggle EXATO
    matches = KAGGLE_REGEX.findall(text)
    if not matches:
        # Tipo A: sem \boxed{}
        results['tipo_a_no_boxed'] += 1
        continue

    # Pega o ULTIMO \boxed{} (Kaggle pega ultimo nao-vazio)
    extracted = None
    for m in reversed(matches):
        if m.strip():
            extracted = m.strip()
            break

    if not extracted:
        results['tipo_a_no_boxed'] += 1
        continue

    # Verify Kaggle EXATO
    is_correct = kaggle_verify(gold, extracted)

    if is_correct:
        results['kaggle_correct'] += 1
        continue

    # Errado: classificar Tipo B vs Tipo C
    # Tipo C: format off (resposta seria certa se reformatasse)
    # Tipo B: math errado (resposta diferente do gold)

    # Tentativas de cleanup para detectar Tipo C:
    cleanup_attempts = [
        extracted.replace('=', ' ').strip().split()[-1] if extracted.split() else '',  # x = 42 -> 42
        re.sub(r'\s+', '', extracted),  # remove espacos
        re.sub(r'[a-zA-Z/\s]', '', extracted),  # remove letters/units
        extracted.rstrip('.,;'),  # remove trailing punct
        extracted.split()[0] if extracted.split() else extracted,  # primeira palavra
        extracted.split()[-1] if extracted.split() else extracted,  # ultima palavra
    ]

    is_format_off = False
    for attempt in cleanup_attempts:
        if attempt and attempt != extracted and kaggle_verify(gold, attempt):
            is_format_off = True
            break

    if is_format_off:
        results['tipo_c_format_off'] += 1
        if len(results['tipo_c_examples']) < 10:
            results['tipo_c_examples'].append({
                'gold': gold[:50],
                'extracted_raw': extracted[:80],
                'fixed_attempt': attempt[:50],
            })
    else:
        results['tipo_b_math_wrong'] += 1

elapsed_total = time.time() - t_start

# ============================================================
# STEP 8: REPORT
# ============================================================
print('\n' + '=' * 70)
print('=== PARSER TEST RESULTS ===')
print('=' * 70)
print(f'Adapter source: {adapter_source}')
print(f'Total evaluated: {results["total"]}')
print(f'Time: {elapsed_total:.0f}s')
print()
print(f'Kaggle parser score: {results["kaggle_correct"]}/{results["total"]} = {100*results["kaggle_correct"]/max(1,results["total"]):.1f}%')
print()
print(f'Error breakdown:')
total_errors = results['total'] - results['kaggle_correct']
print(f'  Tipo A (no \\boxed{{}}):     {results["tipo_a_no_boxed"]:3} ({100*results["tipo_a_no_boxed"]/max(1,results["total"]):.1f}%)')
print(f'  Tipo B (math wrong):           {results["tipo_b_math_wrong"]:3} ({100*results["tipo_b_math_wrong"]/max(1,results["total"]):.1f}%)')
print(f'  Tipo C (FORMAT OFF - parser miss): {results["tipo_c_format_off"]:3} ({100*results["tipo_c_format_off"]/max(1,results["total"]):.1f}%) <- HIPOTESE GEMINI PRO')
print()

if results['tipo_c_examples']:
    print('Tipo C examples (resposta correta perdida por format off):')
    for ex in results['tipo_c_examples']:
        print(f'  gold="{ex["gold"]}" | extracted="{ex["extracted_raw"]}" | seria_correto="{ex["fixed_attempt"]}"')
    print()

# === DECISAO ===
print('=== DECISAO ===')
tipo_c_pct = 100 * results['tipo_c_format_off'] / max(1, results['total'])
tipo_b_pct = 100 * results['tipo_b_math_wrong'] / max(1, results['total'])
tipo_a_pct = 100 * results['tipo_a_no_boxed'] / max(1, results['total'])

if tipo_c_pct >= 5:
    print(f'>>> HIPOTESE GEMINI PRO **CONFIRMADA**: {tipo_c_pct:.1f}% erros sao FORMAT OFF')
    print(f'>>> SOLUCAO REAL: SFT extra ~200-500 steps forcando \\boxed{{X}} formato STRICT')
    print(f'>>> Custo: ~30-60 min train, 1 slot Kaggle, prob 0.87+ = 40-60% se Tipo C >5')
    print(f'>>> NAO VALE: parser fix client-side (Kaggle controla parser)')
elif tipo_b_pct > tipo_c_pct * 2:
    print(f'>>> HIPOTESE GEMINI PRO REJEITADA: {tipo_b_pct:.1f}% erros sao MATH wrong')
    print(f'>>> SOLUCAO REAL: warmstart + SFT amplo com HEM (FASE 3 do roadmap)')
    print(f'>>> Continuar v154 MODE train_warmstart')
elif tipo_a_pct >= 20:
    print(f'>>> ALERTA: {tipo_a_pct:.1f}% sem \\boxed{{}} - max_tokens issue ou prompt')
    print(f'>>> Verificar inference settings (max_tokens, prompt format)')
else:
    print(f'>>> Resultado misto - precisa analise manual')
    print(f'>>> Sugestao: rodar em mais samples (500+) para confidence')

print()
print('=== NEXT STEP ===')
if tipo_c_pct >= 5:
    print('Execute v156 com SFT format-strict (vou criar baseado neste resultado)')
elif tipo_b_pct > tipo_c_pct:
    print('Execute v154 MODE train_warmstart (HEM + 9 fixes)')
else:
    print('Inspect manual results before deciding')

# Save full report
report_path = '/content/v155_parser_report.json'
with open(report_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nFull report saved: {report_path}')
